# Species Accounts RAG — Chunk, Embed, Store in ChromaDB, Retrieve

This notebook builds a small retrieval pipeline over a folder of plant/species text accounts:

1. Point at the folder with the source `.txt` documents.
2. Split each document into fixed-character chunks.
3. Set the embedding model.
4. Connect to a ChromaDB database stored inside `project0001/Species Accounts`, create/open a collection, and ingest the chunks.
5. Run a simple test query and inspect what gets retrieved.

## 1. Folder with text documents

In [1]:
from pathlib import Path
import shutil

# Source folder containing the raw species-account .txt files to ingest.
DOC_FOLDER = Path.home() / 'Downloads' / 'AgenticFolder' / 'project0001' / 'a.Extractions' / 'b.Text from Extraction Documents - Species RAG'

# Where the ChromaDB database itself will live. Per spec, the "Species Accounts"
# folder sits directly inside project0001 (a sibling of a.Extractions, not nested under it).
PROJECT_DIR = Path.home() / 'Downloads' / 'AgenticFolder' / 'project0001'
PERSIST_DIR = PROJECT_DIR / 'DB Species Accounts'
COLLECTION_NAME = 'db_species_accounts'

print(f"Reading text documents from: {DOC_FOLDER}")
print(f"Chroma database will be persisted at: {PERSIST_DIR}")


Reading text documents from: /Users/andrepirex/Downloads/AgenticFolder/project0001/a.Extractions/b.Text from Extraction Documents - Species RAG
Chroma database will be persisted at: /Users/andrepirex/Downloads/AgenticFolder/project0001/DB Species Accounts


## 2. Load the text documents

Read every `.txt` file in `DOC_FOLDER` into memory, keeping the filename so it can be carried through
as metadata (useful later for tracing a retrieved chunk back to its source document).

In [2]:
txt_paths = sorted(DOC_FOLDER.glob('*.txt'))

if not txt_paths:
    raise FileNotFoundError(f"No .txt files found in {DOC_FOLDER}. Check the path above.")

# {filename: full_text}
source_documents = {
    path.name: path.read_text(encoding='utf-8', errors='replace')
    for path in txt_paths
}

print(f"Loaded {len(source_documents)} documents:")
for name, text in source_documents.items():
    print(f"  - {name}  ({len(text):,} chars)")


Loaded 610 documents:
  - open_library_A_Handbook_of_the_British_Flora.txt  (2,148,872 chars)
  - open_library_A_Handbook_of_the_British_Flora_copy.txt  (2,148,872 chars)
  - open_library_A_History_of_British_Birds_William_Yarrell.txt  (914,964 chars)
  - open_library_A_History_of_British_Birds_William_Yarrell_copy.txt  (914,964 chars)
  - open_library_A_History_of_British_Fishes.txt  (765,402 chars)
  - open_library_A_History_of_British_Fishes_copy.txt  (765,402 chars)
  - open_library_A_History_of_the_Birds_of_Europe.txt  (382,931 chars)
  - open_library_A_History_of_the_Birds_of_Europe_copy.txt  (382,931 chars)
  - open_library_A_Manual_of_British_Botany.txt  (1,219,972 chars)
  - open_library_A_Manual_of_British_Botany_copy.txt  (1,219,972 chars)
  - open_library_A_Year_in_the_Fields.txt  (303,847 chars)
  - open_library_A_Year_in_the_Fields_copy.txt  (303,847 chars)
  - open_library_Britain_s_Butterflies.txt  (575,298 chars)
  - open_library_Britain_s_Butterflies_copy.txt  (575,29

## 2a. Load source citation metadata

Each `.txt` file may have a matching `<filename>.meta.json` sidecar sitting next to it, written
upstream (by the extractor notebooks) with real bibliographic data resolved at download time --
things like the true Wikipedia page title/URL, the Internet Archive identifier, DOI, ISBN, or
Europe PMC authors/journal/year. That's genuine data, not something guessed from the filename.

If a `.txt` file has no sidecar (e.g. it was dropped in by hand), it just gets an empty metadata
dict -- downstream code falls back to showing the filename only, rather than fabricating a title
or author for it.

In [3]:
import json as _json

def load_source_metadata(filename: str) -> dict:
    """Read '<DOC_FOLDER>/<filename>.meta.json' if present, else return {}."""
    meta_path = DOC_FOLDER / f'{filename}.meta.json'
    if not meta_path.exists():
        return {}
    try:
        return _json.loads(meta_path.read_text(encoding='utf-8'))
    except Exception as error:
        print(f"Warning: couldn't parse {meta_path.name}: {error}")
        return {}

# {filename: metadata dict} -- built for every loaded .txt, empty dict if no sidecar found.
source_metadata = {filename: load_source_metadata(filename) for filename in source_documents}

with_metadata = sum(1 for m in source_metadata.values() if m)
print(f"{with_metadata} / {len(source_metadata)} documents have a citation metadata sidecar")
for filename, meta in source_metadata.items():
    if meta:
        print(f"  - {filename}: {meta.get('title', '(no title)')}")


326 / 610 documents have a citation metadata sidecar
  - open_library_A_Handbook_of_the_British_Flora.txt: A Handbook of the British Flora
  - open_library_A_History_of_British_Birds_William_Yarrell.txt: A History of British Birds (William Yarrell)
  - open_library_A_History_of_British_Fishes.txt: A History of British Fishes
  - open_library_A_History_of_the_Birds_of_Europe.txt: A History of the Birds of Europe
  - open_library_A_Manual_of_British_Botany.txt: A Manual of British Botany
  - open_library_A_Year_in_the_Fields.txt: A Year in the Fields
  - open_library_Britain_s_Butterflies.txt: Britain's Butterflies
  - open_library_Britain_s_Orchids.txt: Britain's Orchids
  - open_library_British_Birds.txt: British Birds
  - open_library_British_Butterflies.txt: British Butterflies
  - open_library_British_Entomology_John_Curtis.txt: British Entomology (John Curtis)
  - open_library_British_Trees_and_Shrubs.txt: British Trees and Shrubs
  - open_library_Cat_Owner_s_Home_Veterinary_Handbo

## 2b. Clean the text & remove duplicate documents

Two problems showed up in the retrieved chunks:

1. **Raw `\n` / extra whitespace baked into the text.** The source `.txt` files carry line breaks and
   irregular spacing from the original extraction/OCR. Left as-is, that whitespace ends up inside chunks
   and inside the embedded text, which hurts both readability and match quality.
2. **Duplicate documents.** Several files in the folder are literal duplicates or near-duplicates of each
   other (e.g. `..._copy.txt` files, or two different filenames for the same scanned book — you can spot
   these by matching character counts in the step 2 printout). Ingesting the same content twice wastes
   embedding cost/storage and causes duplicate hits at query time (see chunks #1 and #2 in the retrieval
   test above — same text, two different source files).

This step cleans and deduplicates `source_documents` (from step 2) **before** chunking, so everything
downstream — chunks, embeddings, the Chroma collection — is built from the cleaned corpus.

In [4]:
import hashlib
import re


def clean_text(text: str) -> str:
    """Normalize whitespace so newlines/extra spaces from extraction don't leak into chunks.

    - Collapses any run of whitespace (spaces, tabs, \n, \r) into a single space.
    - Strips leading/trailing whitespace.
    This intentionally flattens paragraph breaks too, since the chunking step is a simple
    fixed-character split that doesn't rely on paragraph structure anyway.
    """
    return re.sub(r"\s+", " ", text).strip()


# --- 1. Clean whitespace in every document -------------------------------
cleaned_documents = {
    filename: clean_text(text) for filename, text in source_documents.items()
}

# --- 2. Drop duplicate documents (identical content under different filenames) ---
# Hash the cleaned text so whitespace differences don't hide/create false duplicates.
seen_hashes = {}       # content hash -> filename first seen
duplicates_dropped = []  # (duplicate filename, original filename it matches)
source_documents_clean = {}

for filename, text in cleaned_documents.items():
    content_hash = hashlib.sha256(text.encode("utf-8")).hexdigest()
    if content_hash in seen_hashes:
        duplicates_dropped.append((filename, seen_hashes[content_hash]))
        continue
    seen_hashes[content_hash] = filename
    source_documents_clean[filename] = text

print(f"Documents before cleaning/dedup: {len(source_documents)}")
print(f"Documents after cleaning/dedup:  {len(source_documents_clean)}")
print(f"Duplicates dropped: {len(duplicates_dropped)}")
for dup_file, original_file in duplicates_dropped:
    print(f"  - '{dup_file}' is a duplicate of '{original_file}' (dropped)")


Documents before cleaning/dedup: 610
Documents after cleaning/dedup:  323
Duplicates dropped: 287
  - 'open_library_A_Handbook_of_the_British_Flora_copy.txt' is a duplicate of 'open_library_A_Handbook_of_the_British_Flora.txt' (dropped)
  - 'open_library_A_History_of_British_Birds_William_Yarrell_copy.txt' is a duplicate of 'open_library_A_History_of_British_Birds_William_Yarrell.txt' (dropped)
  - 'open_library_A_History_of_British_Fishes_copy.txt' is a duplicate of 'open_library_A_History_of_British_Fishes.txt' (dropped)
  - 'open_library_A_History_of_the_Birds_of_Europe_copy.txt' is a duplicate of 'open_library_A_History_of_the_Birds_of_Europe.txt' (dropped)
  - 'open_library_A_Manual_of_British_Botany_copy.txt' is a duplicate of 'open_library_A_Manual_of_British_Botany.txt' (dropped)
  - 'open_library_A_Year_in_the_Fields_copy.txt' is a duplicate of 'open_library_A_Year_in_the_Fields.txt' (dropped)
  - 'open_library_Britain_s_Butterflies_copy.txt' is a duplicate of 'open_library_Br

## 3. Chunks — fixed-character splitting

**Chunking strategy: simple fixed-character splitting.**
`CharacterTextSplitter` with `separator=""` ignores sentence/paragraph boundaries entirely and just cuts
every `chunk_size` characters, with `chunk_overlap` characters repeated between consecutive chunks so
context isn't lost right at a cut point.

Settings used here:
- `chunk_size = 400` — characters per chunk. Small enough to keep each chunk focused and cheap to embed,
  large enough to retain a usable amount of context per species account.
- `chunk_overlap = 20` — characters shared between adjacent chunks, so a fact split across a chunk
  boundary still appears in full in at least one chunk.

Each chunk keeps the originating filename and its position in the source document as metadata.

In [5]:
from langchain_text_splitters import CharacterTextSplitter

# --- Chunking settings ---------------------------------------------------
CHUNK_SIZE = 1000       # characters per chunk
CHUNK_OVERLAP = 100     # characters of overlap between consecutive chunks
# ---------------------------------------------------------------------------

fixed_splitter = CharacterTextSplitter(
    separator="",              # no separator: cut anywhere (pure fixed-character splitting)
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

# chunk_records: one dict per chunk, carrying enough metadata to trace it back
# to its source document and its position within that document -- plus,
# where available, real citation fields from that document's .meta.json
# sidecar (see step 2a). Chroma only accepts scalar metadata values
# (str/int/float/bool), so citation fields are flattened onto the chunk
# record individually rather than nested under a "citation" key, and any
# non-scalar or missing value is simply omitted.
chunk_records = []
for filename, text in source_documents_clean.items():
    doc_meta = source_metadata.get(filename, {})
    citation_fields = {
        f"cite_{key}": value
        for key, value in doc_meta.items()
        if isinstance(value, (str, int, float, bool))
    }
    chunks = fixed_splitter.split_text(text)
    for i, chunk_text in enumerate(chunks):
        chunk_records.append(
            {
                "id": f"{filename}::chunk-{i}",
                "text": chunk_text,
                "source": filename,
                "chunk_index": i,
                **citation_fields,
            }
        )

print(f"{len(chunk_records)} total chunks from {len(source_documents)} documents")
print("--- example chunk ---")
print(chunk_records[0]["id"])
print(repr(chunk_records[0]["text"][:200]))


32888 total chunks from 610 documents
--- example chunk ---
open_library_A_Handbook_of_the_British_Flora.txt::chunk-0
', pe Ce 6s Bey) ents Hid ee tte et © Soli iad - or ee ee a ee es ee ee od ala Jy ‘ f > ‘. ~ = = >> x _— ni 5 Poin i Ae a me : ~S ee : aaa i: : ns at o. wnt ae * a ae 4 Salekaieneiet ae bone fed wi Te '


## 4. Embedding model

**Embedding settings.** Using OpenAI's `text-embedding-3-small`: a good low-cost default for a lesson/
prototype-sized corpus. Swap the `model` string here (e.g. to `text-embedding-3-large`) if higher-fidelity
embeddings are needed later — just remember that changing the embedding model changes the vector space,
so any previously-built collection should be rebuilt rather than reused.

Requires `OPENAI_API_KEY` to be set in the environment.

In [6]:
from langchain_openai import OpenAIEmbeddings

# --- Embedding settings ----------------------------------------------------
EMBEDDING_MODEL = "text-embedding-3-small"
# ---------------------------------------------------------------------------

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
print(f"Embedding model set: {EMBEDDING_MODEL}")


Embedding model set: text-embedding-3-small


## 5. ChromaDB connection — "Species Accounts" collection

**Storage settings.**
- The Chroma database lives on disk at `PERSIST_DIR`, i.e. `project0001/Species Accounts`, as required.
- The collection inside that database is named `species_accounts`.
- Ingestion is idempotent: each chunk gets a stable ID (`<filename>::chunk-<index>`), and only chunks not
  already present in the collection get embedded and added. Re-running this cell on an unchanged corpus
  does no extra embedding work.

In [7]:
from langchain_chroma import Chroma
from langchain_core.documents import Document

# Make sure the parent "Species Accounts" folder exists inside project0001.
PERSIST_DIR.mkdir(parents=True, exist_ok=True)

# Opens the collection if it already exists on disk, creates it otherwise.
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(PERSIST_DIR),
)

# Build LangChain Document objects from the chunks, keeping traceable metadata
# -- including any cite_* citation fields carried through from step 3
# (real title/author/url/etc. where a .meta.json sidecar was found).
docs = [
    Document(
        page_content=rec["text"],
        metadata={
            "source": rec["source"],
            "chunk_index": rec["chunk_index"],
            **{k: v for k, v in rec.items() if k.startswith("cite_")},
        },
    )
    for rec in chunk_records
]
stable_ids = [rec["id"] for rec in chunk_records]

# --- Batching setting -------------------------------------------------
# SQLite (which Chroma uses under the hood) caps how many bound variables a
# single query can hold. Passing all IDs to vector_store.get(ids=...) or
# .add_documents(...) in one call can raise:
#   InternalError: ... too many SQL variables
# once the corpus is large enough. Batching keeps each call well under that
# limit. 200 is conservative and safe across SQLite builds; raise it if you
# want fewer round trips and know your SQLite allows more variables.
BATCH_SIZE = 200
# ---------------------------------------------------------------------------

def batched(seq, size):
    for start in range(0, len(seq), size):
        yield seq[start:start + size]

# Idempotent ingestion: ask which IDs are already indexed (in batches), embed only the rest.
present_ids = set()
for id_batch in batched(stable_ids, BATCH_SIZE):
    present_ids.update(vector_store.get(ids=id_batch)["ids"])

missing = [
    (doc_id, doc) for doc_id, doc in zip(stable_ids, docs) if doc_id not in present_ids
]

# Chroma also caps how many records a single add/upsert call accepts
# (its own max batch size, separate from SQLite's variable limit above).
# Batching here avoids: ValueError: Batch size of N is greater than max batch size of ...
if missing:
    missing_ids = [doc_id for doc_id, _ in missing]
    missing_docs = [doc for _, doc in missing]
    for id_batch, doc_batch in zip(batched(missing_ids, BATCH_SIZE), batched(missing_docs, BATCH_SIZE)):
        vector_store.add_documents(documents=doc_batch, ids=id_batch)

# Verify against the store itself, not against what we think we added (also batched).
indexed_ids = set()
for id_batch in batched(stable_ids, BATCH_SIZE):
    indexed_ids.update(vector_store.get(ids=id_batch)["ids"])

print(f"Collection '{COLLECTION_NAME}' ready at: {PERSIST_DIR}")
print(f"Chunks indexed: {len(indexed_ids)}/{len(stable_ids)}")
print(f"Newly added this run: {len(missing)}")


Collection 'db_species_accounts' ready at: /Users/andrepirex/Downloads/AgenticFolder/project0001/DB Species Accounts
Chunks indexed: 32888/32888
Newly added this run: 4646


## 6. Simple retrieval test

Run a sample question through the vector store and inspect the top matching chunks — including which
source document and chunk index each one came from, and the similarity score.

## 6a. Format proper citations

Turns a chunk's `cite_*` metadata into a readable citation string, tailored per `cite_source_type`.
Falls back to just the raw filename when no metadata sidecar was found for that source -- it never
invents a title, author, or year for a source that wasn't actually verified upstream.

In [8]:
def format_citation(metadata: dict) -> str:
    """Build a human-readable citation from a chunk's metadata dict.

    metadata here is doc.metadata from a retrieved chunk, i.e. it has
    "source" / "chunk_index" plus any "cite_*" fields carried through from
    that document's .meta.json sidecar (see steps 2a, 3, and 5).
    """
    source_type = metadata.get("cite_source_type")
    title = metadata.get("cite_title")
    url = metadata.get("cite_url")

    # No sidecar was found for this document -- be honest about that
    # rather than presenting the filename as if it were a real citation.
    if not source_type and not title:
        return metadata["source"]

    if source_type == "wikipedia":
        label = f'Wikipedia, "{title}"' if title else "Wikipedia"
        return f"{label} — {url}" if url else label

    if source_type == "open_library":
        creator = metadata.get("cite_archive_org_creator")
        date = metadata.get("cite_archive_org_date")
        bits = [title or metadata["source"]]
        if creator:
            bits.append(creator)
        if date:
            bits.append(str(date)[:4])  # just the year, if a full date string
        citation = ", ".join(bits)
        notes = metadata.get("cite_manifest_notes")
        if notes:
            citation += f" ({notes})"
        return f"{citation} — {url}" if url else citation

    if source_type in ("europe_pmc", "arxiv", "libgen"):
        author = metadata.get("cite_author")
        year = metadata.get("cite_year")
        journal = metadata.get("cite_journal")
        bits = [b for b in (author, title or metadata["source"], journal, str(year) if year else None) if b]
        # Avoid a doubled "al.." when an author string already ends in a period.
        citation = ". ".join(b.rstrip(".") for b in bits)
        doi = metadata.get("cite_doi")
        if doi:
            citation += f" — https://doi.org/{doi}"
        elif url:
            citation += f" — {url}"
        return citation

    # Any other/unrecognised source_type: show what we have without guessing shape.
    bits = [b for b in (title, metadata.get("cite_author"), metadata.get("cite_year")) if b]
    citation = ", ".join(bits) if bits else metadata["source"]
    return f"{citation} — {url}" if url else citation


In [9]:
# --- Query settings ---------------------------------------------------
TEST_QUERY = "What does the flower of the three-cornered leek look like?"
TOP_K = 5
# ------------------------------------------------------------------------

results = vector_store.similarity_search_with_score(TEST_QUERY, k=TOP_K)

print(f"Query: {TEST_QUERY}\n")
for rank, (doc, score) in enumerate(results, start=1):
    citation = format_citation(doc.metadata)
    print(f"#{rank}  score={score:.4f}  chunk={doc.metadata['chunk_index']}")
    print(f"  Source: {citation}")
    print(repr(doc.page_content[:300]))
    print("-" * 80)


Query: What does the flower of the three-cornered leek look like?

#1  score=0.7891  chunk=0
  Source: Wikipedia, "Allium triquetrum" — https://en.wikipedia.org/wiki/Allium_triquetrum
'Allium triquetrum is a bulbous flowering plant in the genus Allium native to the Mediterranean basin. It is known in English as three-cornered leek or three-cornered garlic, in Australia as angled onion, and in New Zealand as onion weed. Both the English name and the specific epithet triquetrum refe'
--------------------------------------------------------------------------------
#2  score=0.9030  chunk=150
  Source: The British Herbal (John Hill), Hill, John, 1714?-1775, 1756 (18th-century medicinal plant guide.) — https://archive.org/details/b30450950_0002
'e flowers, and afterwards the leaves. As nature has inverted the general order in the growth of this plant, it is proper, in the defeription, we follow her courfe. The footftalks which fupport the flowers are fhort, and very flender : one flower fta